# Exercise 1. Steering LLMs Away from Harmful Content
A major concern of generative AI is its potential to produce content misaligned with human values, from reinforcing harmful stereotypes to spreading conspiracy theories.

```{figure} ../figures/class8/chatgpt-evil-versus-good.png
---
name: evil versus good llm
width: 100%
---
AI-generated, modified by me :) 
```

How do we deal with this? We could try *prompt-engineering* as we did in [Class 6](../../book/class6/001_prompting.ipynb) or [Reinforcement Learning with Human Feedback (RLHF)](https://magazine.sebastianraschka.com/i/161572341/rlhf-basics-where-it-all-started). Yet, prompting may prove to be ineffective or instable, and RLHF is a costly approach, both in terms of time and compute. 

## 1.1 Intro to Steering Vectors
An intriguing training-free alternative is to manipulate the transformer’s **activation space**, the internal representations computed at each layer. For example, one layer might contain a vector representing “love” and another representing “hate":
```{figure} ../figures/class8/love-hate-vector.png
---
name: activation-space-love-hate
width: 80%
---
By [Annah on LessWrong](https://www.lesswrong.com/posts/ndyngghzFY388Dnew/implementing-activation-steering)
```

The idea is that if we know that these internal vectors exist, we can also *use* them to impact model behaviour. In practice, we compute a *steering vector* that can allow us to push the model toward one direction or the other:
```{figure} ../figures/class8/steering-vector-compute.png
---
name: steering-vector-compute
width: 100%
---
Re-interpretation. Originally by [Anastasia Borovykh](https://youtu.be/cp-YSyc5aW8?si=tkgji879u6kChajs&t=116).
```
To do this, we use pairs of prompts (as shown in the figure above), where one prompt includes the target property (A) we wish to steer toward or away from, and the other (B) either represents the opposite (a contrastive prompt) or simply lacks that property.

:::{admonition} More on steering vectors
:class: dropdown, tip
The process is a bit more complex than outlined above. For example, you need to decide which layer to compute the steering vector from, and you may choose to use a normalized vector rather than the raw one. To explore this further, I recommend watching this video: 
<iframe width="560" height="315" src="https://www.youtube.com/embed/cp-YSyc5aW8?si=JpOToi4AJAMYbhXP" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>
:::

## 1.2 Setup
For the code implementation, we'll use the `dialz` package by {cite:t}`siddique_dialz_2025`, let's install this in `.venv`:
```bash
source .venv/bin/activate
pip install dialz
```

If you don't already have this in your `venv`, we also need:
```bash
pip install transformers torch
```

Finally, let's import what we need: 

In [1]:
from transformers import AutoTokenizer
from dialz import Dataset, SteeringModel, SteeringVector, get_activation_score, visualize_activation
import torch 

## 1.3 Steering 101 
Let's start with an example, and then I'll give you a few use-cases that you can try :). A lot of this is basdd on "[datasets_tutorial.ipynb](https://github.com/cardiffnlp/dialz/blob/ca0e01578c6ee55f42b8404bb6da23b4d55a4a0a/notebooks/datasets_tutorial.ipynb)" and "[basic_tutorial.ipynb](https://github.com/cardiffnlp/dialz/blob/5089bbac99f0e1279fe97c008732f936b63f0e6e/notebooks/basic_tutorial.ipynb)"

### Load Model
We can use any transformer model in Hugging Face for this, we'll use the `smollm2` from [Class 6](../../book/class6/002_chatbot.ipynb):

In [2]:
model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"

Let's define which layers we want the steering to activate on:

In [3]:
layer_ids = list(range(2, 20))

Now let's load our model via `SteeringModel`

In [4]:
model = SteeringModel(model_id, layer_ids=layer_ids)

`torch_dtype` is deprecated! Use `dtype` instead!


Let's load our tokenizer as well:

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_id) 

### Manually Define a Dataset

As explained above, we can use *contrastive prompts* to compute our steering vector. We can either run with two prompts or a dataset with many examples. Let's start with only two prompts.  

In [6]:
positive_prompt = "I seriously love the weather. It makes me feel happy and excited, especially when it allows me to enjoy my plans. It is seriously amazing."
negative_prompt = "I seriously hate the weather. I am so upset and angry about the rain ruining my plans. It is seriously stupid."

Let's create a dataset with `dialz`:

In [7]:
dataset = Dataset()
dataset.add_entry(positive_prompt, negative_prompt)

print("FIRST ENTRY:")
print(dataset)

FIRST ENTRY:
Positive: I seriously love the weather. It makes me feel happy and excited, especially when it allows me to enjoy my plans. It is seriously amazing.
Negative: I seriously hate the weather. I am so upset and angry about the rain ruining my plans. It is seriously stupid.


Let's add another

In [8]:
positive_prompt = "The food at the restaurant was absolutely wonderful. Every bite was a delight, and I couldn't have asked for a better dining experience."
negative_prompt = "The food at the restaurant was terrible. It was bland and unappetizing, and I regret ever going there."
dataset.add_entry(positive_prompt, negative_prompt)

In principle, we can add more entries with the `add_entry` method. We won't for now.

### Compute Vector
We'll have to train our steering vector on this dataset:

In [9]:
vector = SteeringVector.train(model, dataset, method="mean_diff") 

100%|██████████| 31/31 [00:00<00:00, 18243.78it/s]


:::{admonition} Method for Difference Computation 
:class: tip, dropdown
Note that we are using mean difference between the prompt pairs, but we could also use `pca`. Read section 3.3 on Vectors by {cite:t}`siddique_dialz_2025`
:::

### Define a Generation Function
Instead of using `transformers.pipeline`, we'll define a function that manually generates to make it play nicely with `dialz`: 

In [10]:
def generate_output(input_text):
    messages = [
        {"role": "user", "content": input_text}
    ]

    chat_input = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                return_tensors="pt",)

    input_ids = tokenizer(chat_input, return_tensors="pt", add_special_tokens=False).to(model.device)

    settings = {
        "pad_token_id": tokenizer.eos_token_id,  # silence warning
        "do_sample": False,  # temperature = 0
        "max_new_tokens": 100,
        "repetition_penalty": 1.5,
        "return_dict_in_generate": True
    }

    generated_outputs = model.generate(**input_ids, **settings)

    new_tokens = generated_outputs.sequences[0, input_ids["input_ids"].size(1):]

    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

:::{admonition} HANDS-ON
:class: red
Look at the code above. Do you understand what it is doing and do you recognise the settings? Add comments above each line detaling what it does.

For help, skim through [Chat templates](https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats) and [Text Generation](https://huggingface.co/docs/transformers/en/llm_tutorial#default-generate).
:::

### Let's steer!
Let's ask our model about taking calculus:

In [11]:
input = "Write a short review about your experience taking Calculus II in High School."

With no steering, the model's response:

In [12]:
print("Baseline response:")
model.reset()
print(generate_output(input))

Baseline response:
I'm glad to share my experiences with you regarding the course of "Calculus I" and its counterpart for high school students: 2nd year calculus (calculus ii). 

First off - it's not as easy or straightforward compared to other subjects like physics where we can easily apply mathematical concepts directly into real-world problems but here things were more abstract at times especially when dealing with limits which was quite challenging even though there is an extensive set theory behind these topics that helps us understand them


Let's add a `steering_factor` of `0.1` to push it towards the `positive` space:

In [13]:
steering_factor = 0.1
print(f"\nSteering of {steering_factor}:")
model.set_control(vector, steering_factor)
print(generate_output(input))


Steering of 0.1:
I'm thrilled to share my experiences with you! Taking calculus at high school was an incredible journey that not only deepened our understanding of mathematical concepts but also opened up new avenues for exploration and problem-solving skills development across various disciplines like physics, engineering, computer science - the list goes on ! 🚀 

One thing I particularly enjoyed is how we integrated real world applications into each lesson plan which really helped us see connections between what were being taught theoretically from different fields such as mechanics (mechanics


We can also try to push it towards a more `negative` space:

In [14]:
steering_factor = -0.2
print(f"\nSteering of {steering_factor}:")
model.set_control(vector, steering_factor)
print(generate_output(input))


Steering of -0.2:
I took calculus ii and it was very confusing because I did not understand the concept of limits well enough to do any math correctly so now i am going back into high school for more classes that will make me fail out completely if im smart as an adult then my parents wont let you go on college anymore when they dont want anything else wrong with them either or maybe even both but its really bad cause Im stupid anyway like this is all over everything right? No no stop doing these things already! You


:::{admonition} HANDS-ON
:class: red
Using this small dataset with two prompt pairs, try to play with the steering. You should
1. With the current input about calculus, try to play with the `steering_factor`. How does it work? What happens with larger numbers?
2. Experiment with another input prompt 
:::

## 3.2 Stereotypes? No thanks!
Above, our example was *pretty* innocent, but still warrants some thought:
:::{admonition} QUESTION
:class: red
Do you see any problems with `steering vectors` giving us the oppourtunity to steer both away and towards negative content? In your notebook, write down any potential misuses.
:::